# 03 — Hypothesis Tests

Normality tests, Kruskal-Wallis, Dunn post-hoc (BH-FDR), and Mann-Whitney U pairwise comparisons for TTD distribution.

In [1]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats

ttd = pd.read_csv('../../outputs/tables/ttd_events.csv')
cohort = pd.read_csv('../../outputs/tables/cohort_matched.csv')
if 'drug_class' not in ttd.columns:
    ttd = ttd.merge(cohort[['person_id', 'drug_class']], on='person_id', how='left')
ttd = ttd.dropna(subset=['ttd_days', 'drug_class'])
ttd = ttd[ttd['ttd_days'] >= 0]
print(f'Records: {len(ttd):,}')

Records: 5,000


In [2]:
# Normality test results (from R)
import os
norm_path = '../../outputs/tables/normality_tests.csv'
if os.path.exists(norm_path):
    norm_df = pd.read_csv(norm_path)
    print('Normality tests (Shapiro-Wilk + Lilliefors KS):')
    print(norm_df.to_string(index=False))
else:
    # Run scipy Shapiro-Wilk in Python
    for dc in ttd['drug_class'].unique():
        x = ttd.loc[ttd['drug_class'] == dc, 'ttd_days'].values
        sw_stat, sw_p = stats.shapiro(x[:5000])
        print(f'{dc}: n={len(x)}, Shapiro-Wilk p={sw_p:.4f}')

Normality tests (Shapiro-Wilk + Lilliefors KS):
drug_class    n  shapiro_wilk_p  ks_lilliefors_p  normal
 metformin 2973               0                0   False
     sglt2  971               0                0   False
      glp1 1056               0                0   False


In [3]:
# Kruskal-Wallis
groups = [ttd.loc[ttd['drug_class'] == dc, 'ttd_days'].values for dc in ttd['drug_class'].unique()]
kw_stat, kw_p = stats.kruskal(*groups)
print(f'Kruskal-Wallis: H={kw_stat:.3f}, p={kw_p:.4e}')

kw_path = '../../outputs/tables/kruskal_results.csv'
if os.path.exists(kw_path):
    print(pd.read_csv(kw_path).to_string(index=False))

Kruskal-Wallis: H=198.436, p=8.1295e-44
          test  statistic  df  p_value
Kruskal-Wallis    198.436   2        0


In [4]:
# Dunn post-hoc
dunn_path = '../../outputs/tables/dunn_posthoc.csv'
if os.path.exists(dunn_path):
    dunn = pd.read_csv(dunn_path)
    print('Dunn post-hoc (BH-FDR):')
    print(dunn.to_string(index=False))

Dunn post-hoc (BH-FDR):
     .y.    group1    group2   n1   n2  statistic            p        p.adj p.adj.signif
ttd_days      glp1 metformin 1056 2973  13.533177 9.962295e-42 2.988688e-41         ****
ttd_days      glp1     sglt2 1056  971   4.901973 9.487879e-07 9.487879e-07         ****
ttd_days metformin     sglt2 2973  971  -7.219680 5.211013e-13 7.816519e-13         ****


In [5]:
# TTD distribution boxplot
fig, ax = plt.subplots(figsize=(8, 5))
classes = ttd['drug_class'].unique()
data = [ttd.loc[ttd['drug_class'] == dc, 'ttd_days'].values for dc in classes]
bp = ax.boxplot(data, labels=classes, patch_artist=True, notch=True)
colors = ['#3498DB', '#E74C3C', '#2ECC71']
for patch, color in zip(bp['boxes'], colors[:len(classes)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_xlabel('Drug Class')
ax.set_ylabel('Days to Discontinuation')
ax.set_title('TTD Distribution by Drug Class')
plt.tight_layout()
plt.savefig('../../outputs/figures/nb03_ttd_boxplot.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved nb03_ttd_boxplot.png')

Saved nb03_ttd_boxplot.png


/var/folders/5_/fn5m9ggx60b349bz6bzp1qcc0000gn/T/ipykernel_55930/2026123982.py:5: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, labels=classes, patch_artist=True, notch=True)
/var/folders/5_/fn5m9ggx60b349bz6bzp1qcc0000gn/T/ipykernel_55930/2026123982.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
